# Notebook 01 — Train Agent A (EfficientNet-B4) on ISIC-2019

This notebook trains **Agent A**, an EfficientNet-B4 skin-lesion classifier, on the ISIC-2019 dataset using a free Kaggle GPU. It is **fully self-contained**: every helper, dataset, loss, and model definition lives inside this notebook (no imports from sibling `config.py` / `dataset.py` / `losses.py`).

## Kaggle setup checklist (do this before Run All)
1. **Attach the dataset**: right sidebar → *Add Input* → search **`andrewmvd/isic-2019`** → Add. It mounts read-only under `/kaggle/input/isic-2019`.
2. **Accelerator**: *Settings → Accelerator → GPU T4 x2* (or P100).
3. **Internet**: *Settings → Internet → ON* (needed for `pip install` of extras and to download the pretrained EfficientNet-B4 weights).
4. **Downstream notebooks**: this is the first notebook, so there is nothing extra to attach here. Notebooks 03/04/05 will attach **this notebook's output** (`agent_a_best.pth`).
5. **Run All** (top menu). Every artifact is written to `/kaggle/working/`.

**Outputs produced:** `/kaggle/working/agent_a_best.pth` (best checkpoint by validation macro-AUC) plus training-curve, confusion-matrix and Grad-CAM++ figures.

## 1. Install extras, imports, and shared helpers
Installs `timm` (upgraded so the EfficientNet ids resolve), `grad-cam`, and `torchmetrics`. Defines the canonical class order, ImageNet normalization constants, `discover_isic()` and `find_file()`, the device, and the writable output directory.

In [ ]:
import sys, subprocess
print('NOTE: Kaggle Internet must be ON (Settings -> Internet -> ON) for pip + pretrained weights.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'timm', 'grad-cam', 'torchmetrics'])

import os, glob, random, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Device:', DEVICE, '| torch:', torch.__version__, '| timm:', timm.__version__)

# ---- Shared constants ----
ISIC_CLASSES = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']  # canonical index order
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224
NUM_CLASSES = 8


def discover_isic(root='/kaggle/input'):
    """Walk /kaggle/input and return (csv_path, image_dir).
    csv_path: the .csv whose header contains ALL 8 class names.
    image_dir: the directory containing the most ISIC_*.jpg/.jpeg files (case-insensitive).
    Robust to nested folders (some mirrors double-nest ISIC_2019_Training_Input/)."""
    csv_path = None
    needed = set(ISIC_CLASSES)
    for dirpath, _dirs, files in os.walk(root):
        for f in files:
            if f.lower().endswith('.csv'):
                p = os.path.join(dirpath, f)
                try:
                    cols = set(pd.read_csv(p, nrows=0).columns)
                except Exception:
                    continue
                if needed.issubset(cols):
                    csv_path = p
                    break
        if csv_path:
            break

    # image_dir = directory holding the most ISIC_*.jpg/.jpeg files
    best_dir, best_count = None, 0
    for dirpath, _dirs, files in os.walk(root):
        c = 0
        for f in files:
            fl = f.lower()
            if fl.startswith('isic_') and (fl.endswith('.jpg') or fl.endswith('.jpeg')):
                c += 1
        if c > best_count:
            best_count, best_dir = c, dirpath

    print('Discovered CSV   :', csv_path)
    print('Discovered images:', best_dir, '(', best_count, 'files )')
    assert csv_path is not None and os.path.exists(csv_path), 'Could not find ISIC GroundTruth CSV under ' + root
    assert best_dir is not None and os.path.isdir(best_dir), 'Could not find ISIC image directory under ' + root
    return csv_path, best_dir


def find_file(filename_substring, search_roots=('/kaggle/input', '/kaggle/working')):
    """Return the first path whose basename contains filename_substring, else None."""
    for r in search_roots:
        if not os.path.isdir(r):
            continue
        for dirpath, _dirs, files in os.walk(r):
            for f in files:
                if filename_substring in f:
                    p = os.path.join(dirpath, f)
                    print('find_file: found', p)
                    return p
    print('find_file: nothing matching "' + filename_substring + '" under', search_roots)
    return None

## 2. Dataset, transforms, stratified split, class weights, and DataLoaders
Reads the one-hot GroundTruth CSV, builds an inline `ISICDataset` (returns `(tensor, label_int, image_path)`), makes a stratified 85/15 train/val split, computes inverse-sqrt-frequency class weights normalized to mean 1, and wires a `WeightedRandomSampler` into the train loader.

In [ ]:
CSV_PATH, IMAGE_DIR = discover_isic()

df = pd.read_csv(CSV_PATH)
# image filename column may be named 'image' (case-insensitive); fall back to first column
img_col = next((c for c in df.columns if c.lower() == 'image'), df.columns[0])
labels = df[ISIC_CLASSES].values.argmax(axis=1).astype(np.int64)
df = df.assign(_label=labels, _image=df[img_col].astype(str))
print('Total samples:', len(df))
print('Class counts :', dict(zip(ISIC_CLASSES, np.bincount(labels, minlength=NUM_CLASSES))))


class ISICDataset(Dataset):
    """Reads the one-hot ISIC-2019 GroundTruth rows. __getitem__ -> (tensor, label_int, image_path)."""
    def __init__(self, frame, image_dir, transform):
        self.frame = frame.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        path = os.path.join(self.image_dir, row['_image'] + '.jpg')
        img = Image.open(path).convert('RGB')
        img = self.transform(img)
        return img, int(row['_label']), path


train_tf = T.Compose([
    T.Resize(256),
    T.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(30),
    T.ColorJitter(0.2, 0.2, 0.2, 0.05),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.1),
])
val_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df['_label'].values, random_state=42)
print('Train:', len(train_df), '| Val:', len(val_df))

train_ds = ISICDataset(train_df, IMAGE_DIR, train_tf)
val_ds = ISICDataset(val_df, IMAGE_DIR, val_tf)

# Class weights = inverse sqrt frequency normalized to mean 1
train_labels = train_df['_label'].values
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(np.float64)
class_counts = np.clip(class_counts, 1, None)
inv_sqrt = 1.0 / np.sqrt(class_counts)
class_weights = inv_sqrt / inv_sqrt.mean()
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print('Class weights:', dict(zip(ISIC_CLASSES, np.round(class_weights, 3))))

# WeightedRandomSampler for the train loader (over-samples minority classes)
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights), replacement=True)

BATCH_SIZE = 32  # lower to 16 if OOM
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
print('Train batches:', len(train_loader), '| Val batches:', len(val_loader))

## 3. Exploratory data analysis
Class-distribution bar chart and a 3x3 grid of sample images (one per class).

In [ ]:
counts_full = np.bincount(labels, minlength=NUM_CLASSES)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(ISIC_CLASSES, counts_full, color=sns.color_palette('viridis', NUM_CLASSES))
ax.set_title('ISIC-2019 class distribution')
ax.set_ylabel('count')
for b, c in zip(bars, counts_full):
    ax.text(b.get_x() + b.get_width() / 2, c, str(int(c)), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=120)
plt.show()

# 3x3 sample grid: one example per class (8 classes), last cell a random extra
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
axes = axes.ravel()
for i in range(9):
    if i < NUM_CLASSES:
        sub = df[df['_label'] == i]
        row = sub.iloc[0]
        title = ISIC_CLASSES[i]
    else:
        row = df.sample(1, random_state=7).iloc[0]
        title = 'random: ' + ISIC_CLASSES[int(row['_label'])]
    p = os.path.join(IMAGE_DIR, row['_image'] + '.jpg')
    axes[i].imshow(Image.open(p).convert('RGB'))
    axes[i].set_title(title, fontsize=10)
    axes[i].axis('off')
plt.suptitle('Sample lesion images', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_grid.png'), dpi=120, bbox_inches='tight')
plt.show()

## 4. Focal loss (class-weighted)
Inline `FocalLoss` with `gamma=2.0` and per-class `alpha` set to the inverse-sqrt class weights to combat the heavy class imbalance.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        if alpha is not None and not torch.is_tensor(alpha):
            alpha = torch.tensor(alpha, dtype=torch.float32)
        self.register_buffer('alpha', alpha if alpha is not None else None)

    def forward(self, logits, targets):
        logp = F.log_softmax(logits, dim=1)
        ce = F.nll_loss(logp, targets, reduction='none')
        pt = torch.exp(-ce)
        focal = (1 - pt) ** self.gamma * ce
        if self.alpha is not None:
            at = self.alpha.to(logits.device)[targets]
            focal = at * focal
        return focal.mean()


criterion = FocalLoss(gamma=2.0, alpha=class_weights_t)
print('FocalLoss ready (gamma=2.0, alpha=class weights).')

## 5. Model + Phase 1 (train classifier head only)
Builds a pretrained EfficientNet-B4 via `timm`, freezes the backbone, and trains only the classifier head for ~5 epochs with `AdamW(lr=1e-3)` and mixed precision (`autocast` + `GradScaler`). Validation reports balanced accuracy and macro one-vs-rest AUC.

In [ ]:
from torchmetrics.classification import MulticlassAUROC

model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

# timm exposes the head via get_classifier(); freeze the backbone, keep the head trainable.
classifier = model.get_classifier()
for p in model.parameters():
    p.requires_grad = False
for p in classifier.parameters():
    p.requires_grad = True
print('Phase 1 trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))
auroc_metric = MulticlassAUROC(num_classes=NUM_CLASSES, average='macro').to(DEVICE)


def run_epoch(loader, optimizer=None):
    """One pass over loader. Returns (loss, balanced_acc, macro_auc, probs, targets)."""
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss, n = 0.0, 0
    all_probs, all_targets = [], []
    for imgs, ys, _paths in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        ys = ys.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train_mode):
            with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
                logits = model(imgs)
                loss = criterion(logits, ys)
            if train_mode:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += loss.item() * imgs.size(0)
        n += imgs.size(0)
        all_probs.append(torch.softmax(logits.float(), dim=1).detach().cpu())
        all_targets.append(ys.detach().cpu())
    probs = torch.cat(all_probs).numpy()
    targets = torch.cat(all_targets).numpy()
    preds = probs.argmax(axis=1)
    bal_acc = balanced_accuracy_score(targets, preds)
    try:
        macro_auc = roc_auc_score(targets, probs, multi_class='ovr', average='macro',
                                  labels=list(range(NUM_CLASSES)))
    except ValueError:
        macro_auc = float('nan')
    return total_loss / max(n, 1), bal_acc, macro_auc, probs, targets


history = {'train_loss': [], 'val_loss': [], 'val_bal_acc': [], 'val_auc': []}

PHASE1_EPOCHS = 5
opt1 = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
for ep in range(1, PHASE1_EPOCHS + 1):
    tr_loss, _, _, _, _ = run_epoch(train_loader, opt1)
    val_loss, val_bal, val_auc, _, _ = run_epoch(val_loader, None)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_bal_acc'].append(val_bal)
    history['val_auc'].append(val_auc)
    print('[P1 %d/%d] train_loss=%.4f val_loss=%.4f val_bal_acc=%.4f val_macroAUC=%.4f'
          % (ep, PHASE1_EPOCHS, tr_loss, val_loss, val_bal, val_auc))

## 6. Phase 2 (fine-tune last blocks + head, with early stopping)
Unfreezes the last `PHASE2_UNFREEZE_BLOCKS` backbone blocks plus the classifier and fine-tunes with `AdamW(lr=1e-5)` + `CosineAnnealingLR`. **Audit fix (1.2c):** the fixed 15-epoch schedule is replaced by up to `PHASE2_MAX_EPOCHS=40` epochs with early stopping (`PATIENCE=8`) on validation macro-AUC; the best checkpoint by macro-AUC is saved to `/kaggle/working/agent_a_best.pth`.

In [ ]:
# Phase 2: unfreeze the last N backbone blocks + classifier head, then fine-tune.
# AUDIT (1.1): the previous version ran a FIXED 15 epochs with no early stopping,
# so it could under- or over-train. AUDIT (1.2c) fix: extend the budget and add
# early stopping on validation macro-AUC. Loss (focal + sqrt-inv-freq weights) and
# the sqrt-inverse WeightedRandomSampler are already in place (audit 1.2a/1.2b) and
# are intentionally left unchanged.
PHASE2_UNFREEZE_BLOCKS = 2   # EfficientNet-B4 has 7 stages (blocks[0..6]); the last 2 are its heaviest.
for p in model.parameters():
    p.requires_grad = False
if hasattr(model, 'blocks'):
    for blk in model.blocks[-PHASE2_UNFREEZE_BLOCKS:]:
        for p in blk.parameters():
            p.requires_grad = True
for p in model.get_classifier().parameters():
    p.requires_grad = True
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Phase 2 trainable params: %d (%.2fM)' % (trainable, trainable / 1e6))
print('  AUDIT 1.2d: if this is < 5M, raise PHASE2_UNFREEZE_BLOCKS to add capacity.')

# Extended fine-tuning budget with early stopping on validation macro-AUC.
PHASE2_MAX_EPOCHS = 40   # was a fixed 15; early stopping below usually halts sooner.
PATIENCE = 8             # stop if val macro-AUC does not improve for this many epochs.
opt2 = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=PHASE2_MAX_EPOCHS)

CKPT_PATH = os.path.join(OUTPUT_DIR, 'agent_a_best.pth')
best_auc = -1.0
epochs_no_improve = 0
last_epoch = 0
for ep in range(1, PHASE2_MAX_EPOCHS + 1):
    last_epoch = ep
    tr_loss, _, _, _, _ = run_epoch(train_loader, opt2)
    val_loss, val_bal, val_auc, _, _ = run_epoch(val_loader, None)
    sched.step()
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_bal_acc'].append(val_bal)
    history['val_auc'].append(val_auc)
    improved = ''
    if not math.isnan(val_auc) and val_auc > best_auc:
        best_auc = val_auc
        epochs_no_improve = 0
        torch.save(model.state_dict(), CKPT_PATH)
        improved = '  <-- saved best'
    else:
        epochs_no_improve += 1
    print('[P2 %d/%d] train_loss=%.4f val_loss=%.4f val_bal_acc=%.4f val_macroAUC=%.4f lr=%.2e no_improve=%d%s'
          % (ep, PHASE2_MAX_EPOCHS, tr_loss, val_loss, val_bal, val_auc,
             opt2.param_groups[0]['lr'], epochs_no_improve, improved))
    if epochs_no_improve >= PATIENCE:
        print('Early stopping at epoch %d (no val macro-AUC gain for %d epochs). Best=%.4f'
              % (ep, PATIENCE, best_auc))
        break

print('Best macro-AUC: %.4f after %d Phase-2 epochs | checkpoint: %s' % (best_auc, last_epoch, CKPT_PATH))
# Fallback: ensure a checkpoint exists even if AUC was NaN throughout.
if not os.path.exists(CKPT_PATH):
    torch.save(model.state_dict(), CKPT_PATH)
    print('Saved final-state checkpoint as fallback.')


## 7. Training curves
Loss, balanced accuracy, and macro-AUC across both phases (dashed line marks the Phase 1 -> Phase 2 boundary).

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
split = PHASE1_EPOCHS + 0.5

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(list(epochs), history['train_loss'], marker='o', label='train')
axes[0].plot(list(epochs), history['val_loss'], marker='o', label='val')
axes[0].axvline(split, color='gray', ls='--', alpha=0.6)
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()

axes[1].plot(list(epochs), history['val_bal_acc'], marker='o', color='tab:green')
axes[1].axvline(split, color='gray', ls='--', alpha=0.6)
axes[1].set_title('Val balanced accuracy'); axes[1].set_xlabel('epoch')

axes[2].plot(list(epochs), history['val_auc'], marker='o', color='tab:purple')
axes[2].axvline(split, color='gray', ls='--', alpha=0.6)
axes[2].set_title('Val macro AUC'); axes[2].set_xlabel('epoch')

plt.suptitle('Agent A training (dashed line = phase 1 -> phase 2)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=120)
plt.show()

## 8. Confusion matrix + per-class metrics
Loads the best checkpoint, runs the validation set, and reports a row-normalized confusion-matrix heatmap and a per-class metric table.

In [ ]:
from sklearn.metrics import classification_report

model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
_, val_bal, val_auc, val_probs, val_targets = run_epoch(val_loader, None)
val_preds = val_probs.argmax(axis=1)
print('Best-checkpoint val balanced acc=%.4f  macro AUC=%.4f' % (val_bal, val_auc))

cm = confusion_matrix(val_targets, val_preds, labels=list(range(NUM_CLASSES)))
cm_norm = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=ISIC_CLASSES, yticklabels=ISIC_CLASSES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Agent A confusion matrix (row-normalized)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=120)
plt.show()

report = classification_report(val_targets, val_preds, labels=list(range(NUM_CLASSES)),
                               target_names=ISIC_CLASSES, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).transpose().round(4)
print(report_df)

## 8b. Per-class validation report (audit item 1.3)
Explicit balanced accuracy and per-class recall on the held-out split, with a flag for any class whose recall is below 0.40, so minority-class behaviour is visible at a glance after a Kaggle run. Reuses `val_targets`/`val_preds` from the best checkpoint above.

In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score, recall_score

CLASS_NAMES = ISIC_CLASSES
bal_acc = balanced_accuracy_score(val_targets, val_preds)
print('Final validation balanced accuracy: %.4f' % bal_acc)
print('\nPer-class report:')
print(classification_report(val_targets, val_preds, labels=list(range(NUM_CLASSES)),
                            target_names=CLASS_NAMES, digits=4, zero_division=0))
print('\nFlag: any class recall below 0.40?')
recalls = recall_score(val_targets, val_preds, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
for cls, rec in zip(CLASS_NAMES, recalls):
    flag = '  <-- LOW' if rec < 0.40 else ''
    print('  %-4s recall=%.4f%s' % (cls, rec, flag))


## 9. Grad-CAM++ explanations
Overlays Grad-CAM++ saliency on 9 validation images so we can sanity-check that the model attends to the lesion. The target layer is resolved robustly (`model.conv_head`, else `model.blocks[-1]`).

In [ ]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Resolve a sensible conv target layer for EfficientNet
if hasattr(model, 'conv_head') and model.conv_head is not None:
    target_layers = [model.conv_head]
elif hasattr(model, 'blocks'):
    target_layers = [model.blocks[-1]]
else:
    target_layers = [list(model.modules())[-3]]
print('Grad-CAM target layer:', target_layers[0].__class__.__name__)

mean = np.array(IMAGENET_MEAN, dtype=np.float32)
std = np.array(IMAGENET_STD, dtype=np.float32)

vis_samples = val_df.sample(9, random_state=123).reset_index(drop=True)
model.eval()
cam = GradCAMPlusPlus(model=model, target_layers=target_layers)

fig, axes = plt.subplots(3, 3, figsize=(11, 11))
axes = axes.ravel()
for i in range(9):
    row = vis_samples.iloc[i]
    p = os.path.join(IMAGE_DIR, row['_image'] + '.jpg')
    pil = Image.open(p).convert('RGB')
    inp = val_tf(pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = int(model(inp).argmax(dim=1).item())
    grayscale = cam(input_tensor=inp, targets=[ClassifierOutputTarget(pred)])[0]
    # de-normalize the input image for display
    rgb = inp[0].detach().cpu().numpy().transpose(1, 2, 0)
    rgb = np.clip(rgb * std + mean, 0, 1).astype(np.float32)
    overlay = show_cam_on_image(rgb, grayscale, use_rgb=True)
    true_idx = int(row['_label'])
    axes[i].imshow(overlay)
    axes[i].set_title('true=%s  pred=%s' % (ISIC_CLASSES[true_idx], ISIC_CLASSES[pred]),
                      fontsize=9, color=('green' if pred == true_idx else 'red'))
    axes[i].axis('off')
plt.suptitle('Grad-CAM++ on validation images', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'gradcam_grid.png'), dpi=120, bbox_inches='tight')
plt.show()

## Outputs and how to use them downstream

All artifacts are written to **`/kaggle/working/`**:
- **`agent_a_best.pth`** — best EfficientNet-B4 weights (by validation macro-AUC). This is the key file.
- `class_distribution.png`, `sample_grid.png`, `training_curves.png`, `confusion_matrix.png`, `gradcam_grid.png` — figures.

### Download
After the run finishes, open the **Output** tab (right sidebar) and download `agent_a_best.pth`, or **Save Version** so the output is preserved as a dataset.

### Attach to notebooks 03/04/05
1. Save a version of this notebook so its output is registered.
2. In the downstream notebook: *Add Input* → *Your Work / Notebook Output* → select this notebook's output. It mounts under `/kaggle/input/<this-notebook-output>/`.
3. In code, locate it with the shared helper:
   ```python
   ckpt = find_file('agent_a_best.pth')
   model = timm.create_model('efficientnet_b4', pretrained=False, num_classes=8)
   model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
   ```
Always recreate the model with `pretrained=False, num_classes=8` before loading the state dict.